#### এই সপ্তাহে Theory ও Math-এর সম্পর্ক

মানচিত্রে একটা জায়গাকে চেনাতে আমরা দুটো সংখ্যা ব্যবহার করি, অক্ষাংশ ও দ্রাঘিমাংশ। জায়গার নাম না জানলেও ওই দুটো সংখ্যা দিয়ে ঠিক সেই জায়গাটা খুঁজে পাওয়া যায়।

এই সপ্তাহের Theory-তে তুমি পড়বে Tokens, Context Window ও Temperature। সমস্যা হলো, model শব্দ বোঝে না, সে শুধু সংখ্যা বোঝে। তাই লেখাকে ছোট ছোট টুকরো (token) করে প্রতিটা token-কে সংখ্যার একটা তালিকা বানিয়ে model-কে দেওয়া হয়, মানচিত্রের জায়গাকে যেমন সংখ্যা দিয়ে চেনানো হয়।

এখানেই Math কাজে আসে। সংখ্যার এই তালিকার নামই Vector, তার প্রতিটা সংখ্যা একেকটা Scalar, আর তালিকায় কয়টা সংখ্যা আছে সেটাই তার Dimension। এই Vector-ই পরে Embedding হয়ে token-এর অর্থ ধরে রাখে, সেটা আমরা Week 03-এ পড়ব।

> **সারকথা:** Token হলো লেখার টুকরো, আর Vector হলো সেই টুকরোর সংখ্যার রূপ।

#### Part 1: AI Theory

#### Tokenization ও Tokenizer

#### আগে একটা গল্প
দেয়াল তুলতে হলে রাজমিস্ত্রি আগে ইট সাজায়। পুরো দেয়াল এক টুকরো পাথরে কাটা হয় না। ছোট ছোট ইট জুড়েই বড় দেয়াল হয়। Computer লেখা পড়ার সময় ঠিক উল্টোটা করে: বড় বাক্যকে ভেঙে ছোট ছোট ইট বানায়।

#### Token আর Tokenizer
লেখা ভাঙার কাজকে বলে **tokenization**। যে program এই ভাঙার কাজ করে, তার নাম **tokenizer** (লেখা-কাটার যন্ত্র)। ভাঙা প্রতিটা টুকরোর নাম **token**। একটা token পুরো শব্দ হতে পারে, শব্দের অংশ হতে পারে, এমনকি একটা চিহ্নও হতে পারে।

#### নম্বর কেন দরকার?
Computer অক্ষর বোঝে না, সে শুধু সংখ্যা বোঝে। তাই tokenizer প্রতিটা টুকরোকে একটা নম্বর দেয়, যার নাম **token ID**। Model এই নম্বরের সারিই পড়ে। উত্তর দেওয়ার সময়ও সে আগে নম্বর বানায়, পরে tokenizer সেই নম্বর আবার লেখায় ফিরিয়ে দেয়।

> "আমি ভাত খাই" → ["আমি", "ভাত", "খাই"] → [0, 1, 2]  
> লেখা → টুকরো → নম্বর

**Demo:** নিচের code-এ একটা বাক্যকে শব্দে ভেঙে প্রতিটাকে একটা নম্বর দেওয়া হয়েছে, তারপর নম্বর থেকে আবার বাক্য ফেরানো হয়েছে।

#### মনে রাখার এক লাইন
> Tokenizer লেখাকে টুকরো করে নম্বর দেয়, আর model সেই নম্বরই বোঝে।

In [1]:
sentence = "আমি ভাত খাই আমি চা খাই"

tokens = sentence.split()
print("tokens:", tokens)

vocab = {}

for token in tokens:
    if token not in vocab:
        vocab[token] = len(vocab)

print("vocab :", vocab)

token_ids = [vocab[token] for token in tokens]
print("ids   :", token_ids)

id_to_token = {number: token for token, number in vocab.items()}

print("ফেরত  :", " ".join(id_to_token[number] for number in token_ids))

tokens: ['আমি', 'ভাত', 'খাই', 'আমি', 'চা', 'খাই']
vocab : {'আমি': 0, 'ভাত': 1, 'খাই': 2, 'চা': 3}
ids   : [0, 1, 2, 0, 3, 2]
ফেরত  : আমি ভাত খাই আমি চা খাই


#### Tokens vs Words

#### আগে একটা গল্প
বাসে উঠলে ভাড়া কীভাবে ঠিক হয়? যাত্রী কতজন, তা দিয়ে নয়। কত কিলোমিটার গেলে, তা দিয়ে। অনেক দূর গেলে ভাড়া বেশি, কাছে গেলে কম।

AI-র খরচও এভাবে মাপা হয়। মাপার একক শব্দ নয়, **token** (লেখার ছোট টুকরো)।

#### ১ token ≠ ১ শব্দ
অনেকে ভাবে, একটা শব্দ মানেই একটা token। এটা ঠিক নয়। ছোট আর চেনা শব্দ একটা token হতে পারে। লম্বা বা অচেনা শব্দ ভেঙে দুই-তিন token হয়।

ইংরেজিতে গড়ে ১টা শব্দ প্রায় **১.৩ token**। বাংলায় সাধারণত আরও বেশি token লাগে, কারণ বাংলা অক্ষর বেশি টুকরো হয়ে ভাঙে। একই কথা বাংলায় লিখলে তাই token বেশি গোনা হয়।

#### এর ফল কী?
Model-এর **খরচ** আর **সীমা**, দুটোই token দিয়ে মাপা হয়। বাংলায় বেশি token লাগলে খরচও কিছুটা বেশি পড়তে পারে, আর সীমার জায়গাও বেশি খরচ হয়। তাই "কত শব্দ" না ভেবে "কত token" ভাবতে হয়।

> বাসের ভাড়া → কিলোমিটারে মাপা হয়, যাত্রীর সংখ্যায় নয়  
> AI-র খরচ → token-এ মাপা হয়, শব্দের সংখ্যায় নয়

#### মনে রাখার এক লাইন
> খরচ ও সীমা মাপা হয় token দিয়ে। ১ token সবসময় ১ শব্দ নয়, আর বাংলায় সাধারণত বেশি token লাগে।

#### Context Window ও Context Length

#### আগে একটা গল্প
তুমি একটা ছোট টেবিলে পড়তে বসেছ। টেবিলে একসাথে পাঁচটা বই ধরে। ছয় নম্বর বই রাখতে চাইলে একটা পুরনো বই সরাতে হবে। যে বই সরালে, তার কথা আর তোমার সামনে নেই। সেটা তুমি এখন দেখতে পাচ্ছ না।

#### Context window
Model-এরও এমন একটা টেবিল আছে। তার নাম **context window**। এটা হলো model একবারে যতটা লেখা সামনে রেখে কাজ করতে পারে, তার সীমা। এই সীমার মাপকে বলে **context length**, আর তা মাপা হয় token দিয়ে।

এই টেবিলে শুধু তোমার প্রশ্নই থাকে না। আগের কথাবার্তা, তুমি যে লেখা দিয়েছ, আর model-এর নিজের লেখা উত্তরও থাকে।

#### সীমা পেরোলে কী হয়?
কথোপকথন লম্বা হয়ে সীমা পেরোলে সবচেয়ে পুরনো অংশ বাদ পড়ে যায়। তখন model আগের কথা আর মনে রাখতে পারে না, যেন সে ভুলে গেছে।

> context window = [ পুরনো কথা | তোমার প্রশ্ন | model-এর উত্তর ]  
> জায়গা ভরে গেলে → একদম পুরনো অংশ বাদ

#### মনে রাখার এক লাইন
> Context window হলো model-এর টেবিলের জায়গা। জায়গা ভরে গেলে পুরনো কথা বাদ পড়ে।

#### Input Tokens ও Output Tokens

#### আগে একটা গল্প
চা-দোকানে গিয়ে তুমি বললে, "এক কাপ দুধ-চা, কম চিনি।" দোকানদার চা বানিয়ে দিল। এখানে দুটো কাজ হলো: তুমি অর্ডার দিলে, আর সে চা বানাল। দুটো কাজের খাটুনি আলাদা।

#### Input আর Output
LLM-এর সাথে কথা বলাও এমন। তুমি যা পাঠাও, তার নাম **input**, আর তার টুকরোগুলো **input tokens**। Model যা লিখে ফেরত দেয়, তার নাম **output**, আর টুকরোগুলো **output tokens**।

Input-এ থাকে তোমার প্রশ্ন, নির্দেশ আর আগের কথাবার্তা। Output-এ থাকে model-এর লেখা উত্তর।

#### দাম আলাদা কেন?
সাধারণত দুটোর দাম আলাদা, আর output-এর দাম input-এর চেয়ে বেশি। কারণ উত্তর লিখতে model-কে একটা একটা করে token বানাতে হয়। সেটা বেশি খাটুনির কাজ। তাই উত্তর ছোট হলে খরচও কমে।

> তুমি পাঠালে → input tokens (দাম তুলনায় কম)  
> model লিখল → output tokens (দাম তুলনায় বেশি)

#### মনে রাখার এক লাইন
> যা পাঠাও তা input, যা পাও তা output। দুটোর দাম আলাদা, সাধারণত output-এর বেশি।

#### Temperature, Top-p ও Sampling

#### আগে একটা গল্প
তোমার প্রিয় চা-দোকানে তুমি রোজ একই চা খাও। স্বাদ জানা, কোনো ঝুঁকি নেই। কোনোদিন সাহস করে নতুন কিছু চাইলে হয়তো দারুণ কিছু পেলে, হয়তো বিস্বাদ। নিরাপদ আর নতুনের মাঝে কতটা সাহস নেবে, সেটা তোমার হাতে।

#### Sampling
আগের topic-এ দেখেছ, model পরের token-এর probability হিসাব করে। এখন প্রশ্ন, সে কোনটা বাছবে? সবসময় সবচেয়ে সম্ভাব্যটা, নাকি কখনো কখনো অন্যটাও? সম্ভাবনা ধরে লটারির মতো বাছাই করার পদ্ধতির নাম **sampling**।

#### Temperature
**Temperature** একটা নিয়ন্ত্রণ-নব, যা বাছাইয়ে কতটা সাহস থাকবে তা ঠিক করে।

> কম temperature → নিরাপদ, প্রায় একই রকম উত্তর  
> বেশি temperature → সৃজনশীল, কিন্তু এলোমেলো

ভেতরে ব্যাপারটা এমন: model প্রতিটা বিকল্পকে একটা **score** (পছন্দের নম্বর) দেয়। **Softmax** হলো সেই পদ্ধতি, যা score-কে probability-তে বদলায়। Temperature এই বদলানোর সময় বিকল্পগুলোর ফারাক বাড়ায় বা কমায়।

#### Top-p
**Top-p** আরেকটা নিয়ন্ত্রণ। এটা সম্ভাবনা বড় থেকে ছোট ক্রমে সাজিয়ে এমন ছোট একটা সেট নেয়, যাদের সম্ভাবনার যোগফল p-এর সমান বা বেশি (যেমন p = 0.9)। বাকিগুলো বাদ, আর বাছাই শুধু ওই সেটের ভেতর থেকে হয়। যেমন চা-দোকানের পুরো মেনু না দেখে শুধু জনপ্রিয় কয়েকটা আইটেম থেকে অর্ডার করা।

উদাহরণ: চার শব্দের সম্ভাবনা 0.5, 0.3, 0.15 ও 0.05 আর p = 0.9। বড় থেকে ছোট ক্রমে যোগ করলে 0.5, তারপর 0.8, তারপর 0.95। তিন নম্বর শব্দে এসে যোগফল প্রথমবার 0.9 পেরোল, তাই সেট হলো প্রথম তিনটা শব্দ, আর 0.05-এর শব্দটা বাদ।

**Demo:** নিচের code-এ একই score-এ temperature 0.5, 1.0 আর 2.0 দিয়ে softmax করে probability দেখানো হয়েছে।

#### `scaled.max()` কেন বাদ দেওয়া হয়
`scaled.max()` বাদ দেওয়া হয় যাতে `np.exp` বড় সংখ্যায় overflow না করে। সব সংখ্যা থেকে একই মান বাদ দিলে অনুপাত বদলায় না, তাই ফলও বদলায় না। শেষে মোট (`exp_values.sum()`) দিয়ে ভাগ করায় সব probability-র যোগফল ১ হয়।

#### মনে রাখার এক লাইন
> Temperature কম হলে নিরাপদ ও একই রকম, বেশি হলে সৃজনশীল ও এলোমেলো। Top-p সম্ভাবনার যোগফল p হওয়া পর্যন্ত সেরা কয়েকটার মধ্যে বাছাইকে সীমিত রাখে।

In [2]:
import numpy as np

words = ["খাই", "খাচ্ছি", "রাঁধি", "ঘুমাই"]

scores = np.array([4.0, 3.0, 2.0, 1.0])


def softmax_with_temperature(scores, temperature):
    scaled = scores / temperature
    exp_values = np.exp(scaled - scaled.max())
    return exp_values / exp_values.sum()


for temperature in [0.5, 1.0, 2.0]:
    probs = softmax_with_temperature(scores, temperature)
    print(f"temperature = {temperature}")
    for word, prob in zip(words, probs):
        print(f"  {word:<8} {prob:.3f} {'█' * int(prob * 40)}")

temperature = 0.5
  খাই      0.865 ██████████████████████████████████
  খাচ্ছি   0.117 ████
  রাঁধি    0.016 
  ঘুমাই    0.002 
temperature = 1.0
  খাই      0.644 █████████████████████████
  খাচ্ছি   0.237 █████████
  রাঁধি    0.087 ███
  ঘুমাই    0.032 █
temperature = 2.0
  খাই      0.455 ██████████████████
  খাচ্ছি   0.276 ███████████
  রাঁধি    0.167 ██████
  ঘুমাই    0.102 ████


#### Model Limitations

#### আগে একটা গল্প
তোমার এক বন্ধু আছে, যে প্রচুর বই পড়েছে। প্রায় সব প্রশ্নের উত্তর সে জানে। কিন্তু তার একটা অভ্যাস আছে। যা জানে না, সেটাও সে "জানি না" বলে না। এমন আত্মবিশ্বাসের সাথে একটা গল্প বানিয়ে বলে যে তুমি বিশ্বাস করে ফেলো। সে মিথ্যুক নয়, শুধু ভুল করে। LLM অনেকটা এমনই।

#### পাঁচটা সীমাবদ্ধতা
**১. Hallucination** (না জেনেও আত্মবিশ্বাসের সাথে ভুল বলা)। ভুল তারিখ, বানানো বইয়ের নাম, অচেনা তথ্য, সবই হতে পারে।

**২. Knowledge cutoff** (জ্ঞানের শেষ তারিখ)। Model একটা নির্দিষ্ট তারিখ পর্যন্ত data দিয়ে শিখেছে। তার পরের খবর সে জানে না, যেন অনেকদিন পাহাড়ে কাটিয়ে আসা কেউ।

**৩. Context সীমা।** একবারে সে সীমিত লেখাই সামনে রাখতে পারে। লম্বা কথোপকথনে পুরনো অংশ ভুলে যায়।

**৪. অঙ্ক ও যুক্তিতে ভুল।** সে পরের token অনুমান করে, হিসাব কষে না। তাই বড় অঙ্ক বা জটিল যুক্তিতে ভুল করতে পারে।

**৫. একই প্রশ্নে ভিন্ন উত্তর।** কারণ sampling। প্রতিবার বাছাইয়ে একটু তফাত হয়।

#### তাহলে কী করবে?
LLM-এর উত্তরকে শেষ কথা ধরবে না। গুরুত্বপূর্ণ তথ্য, বিশেষ করে সংখ্যা আর তারিখ, নিজে যাচাই করে নেবে।

#### মনে রাখার এক লাইন
> LLM শক্তিশালী সহযোগী, কিন্তু ভুলহীন নয়। গুরুত্বপূর্ণ কথা নিজে যাচাই করো।

#### Part 2: Math Track

#### Scalar, Vector ও Dimension

#### আগে একটা গল্প
মুদি দোকানে চাল, ডাল আর তেলের দাম আলাদা আলাদা কাগজে না লিখে একটা তালিকায় লেখা হলো: [৫০, ১২০, ১৮০]। এক কাগজে একটা দাম মানে একটা সংখ্যা। কিন্তু তালিকাটা তিনটা সংখ্যার একটা সারি, আর ওই সারিতে কয়টা সংখ্যা আছে বললেই বোঝা যায় তালিকাটা কত বড়।

#### Scalar, Vector ও Dimension
**Scalar** হলো একটা সাধারণ সংখ্যা, যেমন 5 বা 0.5। **Vector** হলো সংখ্যার একটা সারি বা তালিকা, যেমন [1, 2, 3]। Vector-এ কয়টা সংখ্যা আছে, সেটাই তার **dimension**। [1, 2, 3]-এর dimension ৩, আর [10, 20, 30, 40]-এর ৪।

NumPy-তে `a.shape` লিখলে Vector-এর আকার পাওয়া যায়। `(3,)` মানে তিনটা সংখ্যার একটা সারি, অর্থাৎ dimension ৩।

#### AI-র সাথে সম্পর্ক
Model শব্দ বোঝে না, সে শুধু সংখ্যা বোঝে। তাই প্রতিটা token-কে সংখ্যার একটা তালিকা, অর্থাৎ Vector বানিয়ে দেওয়া হয়। একটা সংখ্যায় token-এর অর্থের শুধু একটা দিক ধরা যায়, যেমন কাউকে শুধু উচ্চতা দিয়ে চেনানো। অনেকগুলো সংখ্যা থাকলে অনেক দিক ধরা যায়, যেমন উচ্চতা, ওজন আর বয়স একসাথে। তাই Vector-এর dimension যত বেশি, অর্থ ধরার জায়গা তত বেশি। আসল model-এ একেকটা token-এর Vector-এ কয়েকশো থেকে কয়েক হাজার সংখ্যা থাকে।

**Demo:** নিচের code-এ দুটো Vector-এর shape দেখা হয়েছে: একটায় ৩টা সংখ্যা, আরেকটায় ৪টা।

#### মনে রাখার এক লাইন
> Scalar একটা সংখ্যা, Vector সংখ্যার তালিকা, আর Dimension হলো তালিকায় কয়টা সংখ্যা আছে। Token-এর অর্থ ধরতে তাকে সংখ্যার তালিকা বানানো হয়।

In [3]:
import numpy as np

a = np.array([1, 2, 3])
b = np.array([10, 20, 30, 40])


print(a.shape)
print(b.shape)


(3,)
(4,)


#### 2D ও 3D Coordinate

#### আগে একটা গল্প
গুপ্তধনের নকশায় লেখা: "বটগাছ থেকে ৩ পা পূর্বে, তারপর ৪ পা উত্তরে।" দুটো সংখ্যা দিয়েই জায়গাটা ঠিক হয়ে যায়। মাটির নিচে কত গভীরে খুঁড়তে হবে, সেই তৃতীয় সংখ্যাও যোগ করলে জায়গা আরও নির্দিষ্ট হয়।

#### Coordinate
দুটো সংখ্যায় সমতলের একটা বিন্দু ঠিক করা যায়, যেমন [3, 4]। একে **2D coordinate** বলে। তিনটা সংখ্যায় ফাঁকা জায়গার (space) একটা বিন্দু ঠিক হয়, যেমন [3, 4, 5]। একে **3D coordinate** বলে। শুরুর বিন্দু (০ বিন্দু) থেকে ওই বিন্দু পর্যন্ত একটা তীর টানলে সেটাই Vector-এর ছবি: তীরটা কতটা লম্বা আর কোন দিকে, তা Vector-এর সংখ্যাগুলোই ঠিক করে দেয়।

চারটা বা তার বেশি সংখ্যা হলে ছবি আঁকা যায় না, কিন্তু হিসাব একই নিয়মে চলে।

#### AI-র সাথে সম্পর্ক
Token-এর Vector আসলে অনেক dimension-এর একটা জায়গার বিন্দু। কাছাকাছি অর্থের token-এর বিন্দু কাছাকাছি পড়ে। Week 03-এ Embedding-এ এটাই দেখবে।

#### মনে রাখার এক লাইন
> Coordinate হলো বিন্দুর সংখ্যার ঠিকানা, আর Vector হলো ০ বিন্দু থেকে সেই ঠিকানা পর্যন্ত টানা তীর।

#### Vector-এর যোগ, বিয়োগ ও Scalar গুণ

#### আগে একটা গল্প
দোকানে সকালে বিক্রি হলো [২ কেজি চাল, ৪ কেজি ডাল, ৬ কেজি চিনি], আর বিকেলে [১, ১, ১]। দিনের মোট বিক্রি বের করতে চালের সাথে চাল, ডালের সাথে ডাল, চিনির সাথে চিনি যোগ করা হয়। আবার আগামী সপ্তাহে সব জিনিস তিনগুণ বিক্রি হলে তিনটা সংখ্যাকেই ৩ দিয়ে গুণ করা হয়।

#### তিনটা কাজ
**যোগ ও বিয়োগ:** একই জায়গার সংখ্যা ধরে ধরে হয়, তাই দুই Vector-এর dimension সমান লাগে। [2, 4, 6] + [1, 1, 1] = [3, 5, 7], আর [2, 4, 6] − [1, 1, 1] = [1, 3, 5]।

**Scalar গুণ:** একটা সংখ্যা (scalar) দিয়ে গুণ করলে Vector-এর প্রতিটা সংখ্যা সেই সংখ্যা দিয়ে গুণ হয়। 3 × [2, 4, 6] = [6, 12, 18], আর 0.5 × [2, 4, 6] = [1, 2, 3]। ধনাত্মক সংখ্যা দিয়ে গুণ করলে Vector একই দিকে থাকে, শুধু লম্বা বা খাটো হয়।

#### AI-র সাথে সম্পর্ক
Model-এর ভেতরে token-এর Vector-এর সাথে অন্য তথ্যের Vector যোগ করা হয়। Week 03-এ Positional Encoding-এ শব্দের Vector-এর সাথে তার জায়গার Vector ঠিক এভাবেই যোগ হয়েছে। আর Vector-কে ছোট বা বড় করতে scalar গুণ লাগে।

**Demo:** নিচের code-এ চারটা কাজই করা হয়েছে।

#### মনে রাখার এক লাইন
> Vector যোগ-বিয়োগ ঘরে ঘরে হয় ও dimension সমান লাগে, আর scalar গুণ প্রতিটা সংখ্যাকে একই সংখ্যা দিয়ে গুণ করে।

In [4]:
a = np.array([2, 4, 6])
b = np.array([1, 1, 1])

print(a + b)
print(a - b)
print(3 * a)
print(0.5 * a)

[3 5 7]
[1 3 5]
[ 6 12 18]
[1. 2. 3.]


#### Vector-এর Magnitude (দৈর্ঘ্য)

#### আগে একটা গল্প
মাঠে একজন সোজা ৩ পা পূর্বে হাঁটল, তারপর ৪ পা উত্তরে গেল। শুরুর জায়গা থেকে সে এখন সোজা কতটা দূরে? মোট ৭ পা হাঁটলেও সোজা দূরত্ব ৫ পা। কোণা ঘুরে না গিয়ে সোজা গেলে পথ কম লাগে।

#### Magnitude
Vector-এর **magnitude** হলো ০ বিন্দু থেকে তার বিন্দু পর্যন্ত সোজা দূরত্ব, অর্থাৎ তীরের দৈর্ঘ্য। প্রতিটা সংখ্যার বর্গ করে সব যোগ করা হয়, তারপর তার বর্গমূল নেওয়া হয়। [3, 4]-এর জন্য:

> √(3² + 4²) = √(9 + 16) = √25 = 5

NumPy-তে `np.linalg.norm(v)` লিখলে magnitude পাওয়া যায়।

#### AI-র সাথে সম্পর্ক
Magnitude বলে দেয় একটা Vector কতটা "বড়"। দুটো Vector কতটা কাছাকাছি বা কতটা মেলে, তা মাপার পদ্ধতিগুলো (W04 Math) এই দৈর্ঘ্যের ওপর দাঁড়িয়ে আছে।

**Demo:** নিচের code ব্যবহারকারীর কাছ থেকে সংখ্যা নেয়। এখানে "3 4" দেওয়া হবে, মানে Vector [3, 4], আর ফল 5.0।

#### মনে রাখার এক লাইন
> Magnitude হলো Vector-এর দৈর্ঘ্য: প্রতিটা সংখ্যার বর্গ যোগ করে বর্গমূল নেওয়া।

Run All-এ `input()` আটকে যায়, তাই নিচের cell উত্তর "3 4" নিজেই দেয়, আর তার পরের `del input` cell আসল `input()` ফিরিয়ে আনে।

In [ ]:
_scripted_answers = iter(["3 4"])

def input(prompt=""):
    answer = next(_scripted_answers)
    print(f"{prompt}{answer}")
    return answer


In [5]:
v = np.array(input("enter numbers").split(), dtype=float)

print(np.linalg.norm(v))

5.0


In [ ]:
del input


#### Vector যোগ করে তার Magnitude

#### আগে একটা গল্প
আগের লোকটা আবার হাঁটল। প্রথমে [3, 1], অর্থাৎ ৩ পা পূর্বে আর ১ পা উত্তরে। তারপর [1, 2], অর্থাৎ ১ পা পূর্বে আর ২ পা উত্তরে। শুরুর জায়গা থেকে সে এখন সোজা কতটা দূরে? আগে দুই ধাপ যোগ করে মোট সরে যাওয়া বের করতে হয়, তারপর সেই যোগফলের দৈর্ঘ্য মাপতে হয়।

#### যোগফলের Magnitude
[3, 1] + [1, 2] = [4, 3], আর তার magnitude √(4² + 3²) = √25 = 5।

এখানে একটা জিনিস লক্ষ করার মতো। আলাদা আলাদা দৈর্ঘ্য √10 ≈ 3.16 আর √5 ≈ 2.24, যোগ করলে ≈ 5.40 হয়। কিন্তু যোগফলের দৈর্ঘ্য ৫। তাই **দুই Vector-এর দৈর্ঘ্যের যোগফল আর যোগফলের দৈর্ঘ্য সাধারণত সমান নয়**। দুই ধাপ ঠিক একই দিকে না হলে যোগফলের দৈর্ঘ্য কম পড়ে।

#### AI-র সাথে সম্পর্ক
Embedding-এর সাথে position-এর Vector যোগ করলে ফল আবার একটা Vector হয়, আর তার নিজের একটা দৈর্ঘ্য থাকে। সেই দৈর্ঘ্য জানতে হলে আগে Vector যোগ করে নিতে হয়, তারপর মাপতে হয়।

**Demo:** নিচের code-এ আগে a + b প্রিন্ট হয়েছে, তারপর তার magnitude।

#### মনে রাখার এক লাইন
> যোগফলের Magnitude জানতে আগে Vector যোগ করো, তারপর দৈর্ঘ্য মাপো। দৈর্ঘ্যগুলো আলাদা যোগ করলে ভুল ফল আসে।

In [6]:
a = np.array([3, 1])
b = np.array([1, 2])

print(a+b)
print(np.linalg.norm(a+b))

[4 3]
5.0


#### Week-end Gate

নিজে উত্তর ভেবে তারপর নিচের উত্তর মেলাও।

**প্রশ্ন ১:** [5, 1, 7, 2] Vector-এর dimension কত?

**প্রশ্ন ২:** [1, 2] + [3, 4] কত?

**প্রশ্ন ৩:** 3 × [2, 0, 1] কত?

**প্রশ্ন ৪:** [6, 8] Vector-এর magnitude কত?

<details>
<summary>উত্তর দেখতে click করো</summary>

১. **৪।** চারটা সংখ্যা আছে, তাই dimension ৪।

২. **[4, 6]।** 1 + 3 = 4, আর 2 + 4 = 6।

৩. **[6, 0, 3]।** 3×2 = 6, 3×0 = 0, 3×1 = 3।

৪. **১০।** √(6² + 8²) = √(36 + 64) = √100 = 10।

</details>

নিচের cell-এ চারটা উত্তর NumPy দিয়ে মিলিয়ে দেখা যায়।

In [ ]:
print(np.array([5, 1, 7, 2]).shape)

print(np.array([1, 2]) + np.array([3, 4]))

print(3 * np.array([2, 0, 1]))

print(np.linalg.norm(np.array([6, 8])))
